In [ ]:
import sys
from pathlib import Path

p = Path().resolve()

# шукаємо root проекту (там де є tools)
while p != p.parent:
    if (p / "tools").exists():
        sys.path.insert(0, str(p))
        break
    p = p.parent

from tools.course_ui import activate
activate()


# Урок 5: Модулі, Імпорти та Консольні програми (CLI) в Python

## ⚠️ Перед початком

1. Виконуйте клітинки **зверху вниз**
2. Спочатку запустіть системну клітинку 🔒
3. Введіть своє ім'я та натисніть **Підтвердити**
4. Для кожного завдання: зробіть прогноз → натисніть Результат

In [ ]:
# @title 🔒 Натисніть Run ОДИН раз перед уроком
# 🔒 system cell (не змінювати)

import uuid, time, requests
from tools.client import API_URL as _API_URL

_LESSON_ID = "lesson_05_exam"

SYSTEM_READY    = True
COMPLETED_TASKS = set()
_SESSION_TOKEN  = None

try:
    STUDENT_NAME
except NameError:
    STUDENT_NAME = None


def require_system():
    if "SYSTEM_READY" not in globals():
        print("⛔ Спочатку натисніть:")
        print("👉 🔒 Натисніть Run ОДИН раз перед уроком")
        return False
    return True


def require_student():
    if not require_system():
        return False
    if not STUDENT_NAME:
        print("⛔ Спочатку представтесь 👇")
        print("👉 Запустіть клітинку '👤 Представтесь'")
        return False
    return True


def _start_session(name):
    global _SESSION_TOKEN
    try:
        r = requests.get(_API_URL, params={"name": name, "lesson_id": _LESSON_ID}, timeout=10)
        data = r.json()
        if data.get("ok"):
            _SESSION_TOKEN = data["token"]
            return True
        return False
    except Exception:
        return False


def submit_task(name, lesson, task):
    key = f"{lesson}_{task}"
    if key in COMPLETED_TASKS:
        print("⚠️ Завдання вже зараховано")
        return
    if not require_student():
        return
    if _SESSION_TOKEN is None:
        COMPLETED_TASKS.add(key)
        print("✅ Завдання зараховано (локально)")
        return
    try:
        r = requests.post(_API_URL, json={
            "token":     _SESSION_TOKEN,
            "task_id":   task,
            "result":    {"completed": True},
            "nonce":     str(uuid.uuid4()),
            "client_ts": int(time.time() * 1000),
        }, timeout=10)
        data = r.json()
        COMPLETED_TASKS.add(key)
        if data.get("ok"):
            print("✅ Завдання зараховано")
        else:
            print("✅ Завдання зараховано (локально)")
    except Exception:
        COMPLETED_TASKS.add(key)
        print("✅ Завдання зараховано (локально)")


print("🔧 System ready:", SYSTEM_READY)


## 👤 Крок 0 — представтесь

Вкажіть своє ім'я перед початком уроку.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

_name_w = widgets.Text(
    placeholder="Ваше ім'я",
    description="Ім'я:",
    layout=widgets.Layout(width="320px"))
_name_btn = widgets.Button(description="Підтвердити", button_style="primary")
_name_out = widgets.Output()

def _on_name_click(b):
    global STUDENT_NAME
    name = _name_w.value.strip()
    if not name:
        with _name_out:
            clear_output()
            print("⚠️ Введіть ім'я")
        return
    STUDENT_NAME = name
    _name_btn.disabled = True
    with _name_out:
        clear_output()
        print("⏳ Підключення до сервера...")
    ok = _start_session(name)
    with _name_out:
        clear_output()
        if ok:
            print(f"✅ Привіт, {STUDENT_NAME}! Сесія розпочата.")
        else:
            print(f"✅ Привіт, {STUDENT_NAME}! (офлайн-режим)")

_name_btn.on_click(_on_name_click)
display(widgets.VBox([_name_w, _name_btn, _name_out]))


---
# Завдання

Кожне завдання складається з трьох кроків:
1. **Подивіться** на код
2. **Запишіть прогноз** у поле вводу
3. **Натисніть кнопку** — Python покаже реальний результат і пояснення

---
## 📦 Частина 1: Механізм імпорту

Коли ви пишете `import something`, Python **читає файл `something.py` зверху донизу і одразу виконує весь код на верхньому рівні**.

Це означає:
- `def` і `class` — **виконуються** (функція/клас створюється в пам'яті), але код всередині них — ні
- Звичайні виклики `print()`, присвоєння змінних — **виконуються одразу**

```python
# greeter.py
print("👋 Модуль greeter завантажено!")  # ← виконається при import!

GREETING = "Привіт"                       # ← теж виконається

def greet(name):                          # ← функція створюється, але не викликається
    return f"{GREETING}, {name}!"
```

# ✅ Task 1 — Побічний ефект при імпорті

### Що виведе Python під час виконання `import greeter`?

Ось вміст файлу `greeter.py`:

```python
print("👋 Модуль greeter завантажено!")

GREETING = "Привіт"

def greet(name):
    return f"{GREETING}, {name}!"
```

> ✏️ Запишіть **точно** що виведе Python в момент `import greeter`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prediction_import = ""

_pt1_w   = widgets.Text(placeholder="Що виведе Python?", description="Прогноз:",
                         layout=widgets.Layout(width="420px"))
_pt1_btn = widgets.Button(description="Зберегти прогноз", button_style="primary")
_pt1_out = widgets.Output()

def _on_pt1(b):
    global prediction_import
    if not require_student():
        with _pt1_out:
            clear_output(); print("⛔ Спочатку представтесь")
        return
    val = _pt1_w.value.strip()
    if not val:
        with _pt1_out:
            clear_output(); print("⚠️ Введіть прогноз")
        return
    prediction_import = val
    _pt1_btn.disabled = True
    with _pt1_out:
        clear_output(); print("✅ Прогноз збережено:", prediction_import)

_pt1_btn.on_click(_on_pt1)
display(widgets.VBox([_pt1_w, _pt1_btn, _pt1_out]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import sys, io

_rev1_btn = widgets.Button(description="🐍 Результат та пояснення", button_style="info")
_rev1_out = widgets.Output()

def _on_rev1(b):
    if not require_student():
        with _rev1_out: print("⛔ Спочатку представтесь")
        return
    if not prediction_import.strip():
        with _rev1_out: print("⚠️ Спочатку зробіть прогноз")
        return
    b.disabled = True
    with _rev1_out:
        print(f"{STUDENT_NAME}, твій прогноз: {prediction_import}")

        # Примусово видаляємо з кешу, щоб побачити print при повторному запуску
        if "greeter" in sys.modules:
            del sys.modules["greeter"]

        # Перехоплюємо stdout
        _buf = io.StringIO()
        _prev = sys.stdout
        sys.stdout = _buf
        import greeter  # noqa
        sys.stdout = _prev
        captured = _buf.getvalue().strip()

        print("\n🐍 Python вивів під час import greeter:")
        print(f"  >>> {repr(captured)}")

        correct = "👋 Модуль greeter завантажено!"
        if prediction_import.strip() == correct:
            print("\n✅ Правильний прогноз!")
        else:
            print(f"\n❌ Неправильно. Очікувалось: {correct}")

        display(Markdown("""
---

## ✅ Що сталося?

**Python читає файл `greeter.py` зверху донизу і виконує весь код верхнього рівня.**

```python
print("👋 Модуль greeter завантажено!")  # ← виконується одразу!
GREETING = "Привіт"                       # ← теж виконується
def greet(name): ...                      # ← функція СТВОРЮЄТЬСЯ (але не викликається)
```

### Проблема
Кожен `import greeter` буде виводити це повідомлення. Якщо ви імпортуєте модуль у 10 місцях — повідомлення з'явиться 10 разів (при першому запуску, тому що Python кешує модулі).

### Рішення — `if __name__ == "__main__":`
Обгорніть весь виконуваний код у цей захисний блок — ми побачимо це в Task 2.

## 🔑 Правило
> Не розміщуйте `print()` та інші виклики на верхньому рівні модуля.  
> Тільки: `def`, `class`, константи і `import`.
"""))

        try:
            submit_task(STUDENT_NAME, _LESSON_ID, "Task_import_side_effects")
        except Exception:
            print("ℹ️ submit_task() не підключений")

_rev1_btn.on_click(_on_rev1)
display(widgets.VBox([_rev1_btn, _rev1_out]))

---
## 🛡️ Частина 2: Захист через `__name__`

Спеціальна змінна `__name__` дозволяє файлу поводитись по-різному залежно від контексту:

| Контекст | Значення `__name__` |
|----------|--------------------|
| Запуск напряму: `python my_math.py` | `"__main__"` |
| Імпорт: `import my_math` | `"my_math"` |

```python
# my_math.py
def add(a, b):
    return a + b

if __name__ == "__main__":    # ← цей блок — ТІЛЬКИ при прямому запуску
    print("Тест:", add(2, 3))
```

Ця конструкція дозволяє файлу бути **і бібліотекою, і програмою** одночасно.

# ✅ Task 2 — Захист `__name__ == "__main__"`

### Що виведе Python під час виконання `import my_math`?

Ось вміст файлу `my_math.py`:

```python
def add(a, b):
    return a + b

def square(x):
    return x * x

if __name__ == "__main__":
    print("=== Тестування my_math ===")
    print("add(2, 3) =", add(2, 3))
```

> ✏️ Що виведе Python в момент `import my_math`? Якщо нічого — введіть `нічого`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prediction_name = ""

_pt2_w   = widgets.Text(placeholder="Що виведе Python?", description="Прогноз:",
                         layout=widgets.Layout(width="420px"))
_pt2_btn = widgets.Button(description="Зберегти прогноз", button_style="primary")
_pt2_out = widgets.Output()

def _on_pt2(b):
    global prediction_name
    if not require_student():
        with _pt2_out:
            clear_output(); print("⛔ Спочатку представтесь")
        return
    val = _pt2_w.value.strip()
    if not val:
        with _pt2_out:
            clear_output(); print("⚠️ Введіть прогноз")
        return
    prediction_name = val
    _pt2_btn.disabled = True
    with _pt2_out:
        clear_output(); print("✅ Прогноз збережено:", prediction_name)

_pt2_btn.on_click(_on_pt2)
display(widgets.VBox([_pt2_w, _pt2_btn, _pt2_out]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import sys, io

_rev2_btn = widgets.Button(description="🐍 Результат та пояснення", button_style="info")
_rev2_out = widgets.Output()

def _on_rev2(b):
    if not require_student():
        with _rev2_out: print("⛔ Спочатку представтесь")
        return
    if not prediction_name.strip():
        with _rev2_out: print("⚠️ Спочатку зробіть прогноз")
        return
    b.disabled = True
    with _rev2_out:
        print(f"{STUDENT_NAME}, твій прогноз: {prediction_name}")

        if "my_math" in sys.modules:
            del sys.modules["my_math"]

        _buf = io.StringIO()
        _prev = sys.stdout
        sys.stdout = _buf
        import my_math  # noqa
        sys.stdout = _prev
        captured = _buf.getvalue().strip()

        print("\n🐍 Під час 'import my_math' Python вивів:")
        if captured:
            print(f"  >>> {repr(captured)}")
        else:
            print("  >>> (нічого)")

        print("\n✍️ А ось що буде, якщо запустити напряму (python my_math.py):")
        print("  === Тестування my_math ===")
        print("  add(2, 3)      =", my_math.add(2, 3))
        print("  multiply(4, 5) =", my_math.multiply(4, 5))
        print("  square(7)      =", my_math.square(7))

        pred_lower = prediction_name.strip().lower()
        if pred_lower in ("нічого", "nothing", "") or not captured:
            if not captured:
                print("\n✅ Правильний прогноз!")
            else:
                print("\n❌ Неправильно. Очікувалось: нічого")
        else:
            print("\n❌ Неправильно. Захист спрацював — import не виводить нічого.")

        display(Markdown("""
---

## ✅ Що сталося?

При `import my_math` Python встановлює `__name__ = "my_math"` — **не** `"__main__"`.

Тому умова `if __name__ == "__main__":` — **False**, і блок тестування не виконується.

| Контекст | `__name__` | Чи виконується блок? |
|----------|-----------|----------------------|
| `python my_math.py` | `"__main__"` | ✅ Так |
| `import my_math` | `"my_math"` | ❌ Ні |

### Результат
Функції `add()`, `multiply()`, `square()` — доступні після імпорту:
```python
import my_math
print(my_math.add(10, 5))    # → 15
print(my_math.square(4))     # → 16
```

## 🔑 Правило
> Завжди захищайте тест-код і точку входу через `if __name__ == "__main__":`.  
> Це дозволяє файлу бути **і модулем, і програмою** водночас.
"""))

        try:
            submit_task(STUDENT_NAME, _LESSON_ID, "Task_name_guard")
        except Exception:
            print("ℹ️ submit_task() не підключений")

_rev2_btn.on_click(_on_rev2)
display(widgets.VBox([_rev2_btn, _rev2_out]))

---
## 🔍 Частина 3: Де Python шукає модулі (`sys.path`)

Коли ви пишете `import random`, Python перевіряє місця у **строгому порядку**:

1. **Поточний каталог** (де знаходиться ваш скрипт)
2. `PYTHONPATH` (змінна середовища)
3. Стандартні бібліотеки Python
4. Встановлені пакети (`site-packages`)

Цей порядок зберігається у списку `sys.path`.

### ⚠️ Import Shadowing — два рівні небезпеки

**Python < 3.13:** локальний `random.py` перекриває стандартну бібліотеку → `AttributeError`  
**Python 3.13+:** stdlib захищена — `random` завантажиться правильно  
**Будь-яка версія:** ваші власні модулі (`myutils.py`) **досі можуть бути затінені!**

```
my_project/
    utils/
        myutils.py   ← "неповна" версія (небезпека!)
    myutils.py       ← повноцінна версія
    main.py          ← який myutils завантажиться?  Залежить від sys.path!
```

❌ **Небезпечні імена для файлів (Python < 3.13):**  
`random.py`, `math.py`, `sys.py`, `os.py`, `string.py`, `time.py`, `json.py`...

❌ **Завжди небезпечно:** дублювати імена власних модулів у різних підпапках.

---
## ⌨️ Частина 4: Аргументи командного рядка (`sys.argv`)

Список `sys.argv` зберігає **аргументи, передані програмі при запуску**:

```bash
python script.py Alice 42
#       ↑          ↑    ↑
#  sys.argv[0]  [1]  [2]
```

| Індекс | Значення |
|--------|----------|
| `sys.argv[0]` | Завжди ім'я скрипта |
| `sys.argv[1]` | Перший аргумент |
| `sys.argv[2]` | Другий аргумент |

⚠️ Всі аргументи — **рядки** (`str`), навіть якщо виглядають як числа!

```python
import sys

if __name__ == "__main__":
    if len(sys.argv) > 1:
        print(f"Привіт, {sys.argv[1]}!")
    else:
        print("Помилка: вкажіть ім'я як аргумент.")
```

# ✅ Task 4 — Аргументи командного рядка

### Що виведе Python?

```python
import sys

args = sys.argv
print("Кількість аргументів:", len(args))
print("Тип sys.argv[1]:", type(args[1]))
print("Сума:", args[1] + args[2])
```

Запуск: `python script.py 10 20`

> ✏️ Що виведе рядок `print("Сума:", args[1] + args[2])`?

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prediction_argv = ""

_pt4_w   = widgets.Text(placeholder="Що виведе print(Сума:...)?",
                         description="Прогноз:", layout=widgets.Layout(width="420px"))
_pt4_btn = widgets.Button(description="Зберегти прогноз", button_style="primary")
_pt4_out = widgets.Output()

def _on_pt4(b):
    global prediction_argv
    if not require_student():
        with _pt4_out:
            clear_output(); print("⛔ Спочатку представтесь")
        return
    val = _pt4_w.value.strip()
    if not val:
        with _pt4_out:
            clear_output(); print("⚠️ Введіть прогноз")
        return
    prediction_argv = val
    _pt4_btn.disabled = True
    with _pt4_out:
        clear_output(); print("✅ Прогноз збережено:", prediction_argv)

_pt4_btn.on_click(_on_pt4)
display(widgets.VBox([_pt4_w, _pt4_btn, _pt4_out]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import sys

_rev4_btn = widgets.Button(description="🐍 Результат та пояснення", button_style="info")
_rev4_out = widgets.Output()

def _on_rev4(b):
    if not require_student():
        with _rev4_out: print("⛔ Спочатку представтесь")
        return
    if not prediction_argv.strip():
        with _rev4_out: print("⚠️ Спочатку зробіть прогноз")
        return
    b.disabled = True
    with _rev4_out:
        print(f"{STUDENT_NAME}, твій прогноз: {prediction_argv}")

        # Симулюємо: args[1] = "10", args[2] = "20"
        args = ["script.py", "10", "20"]

        print("\n🐍 Python виконує:")
        print("  Кількість аргументів:", len(args))
        print("  Тип sys.argv[1]:     ", type(args[1]))
        result = args[1] + args[2]
        print("  Сума:                ", result)

        correct = "Сума: 1020"
        if "1020" in prediction_argv:
            print("\n✅ Правильний прогноз!")
        else:
            print(f"\n❌ Неправильно. args[1] + args[2] = '10' + '20' = '1020' (конкатенація рядків!)")

        print("\n📋 Ваш реальний sys.argv у Jupyter:")
        print(" ", sys.argv)

        display(Markdown("""
---

## ✅ Що сталося?

Усі аргументи в `sys.argv` — це **рядки** (`str`), навіть `"10"` і `"20"`.

```python
args[1] + args[2]  # → "10" + "20" = "1020"  (конкатенація!)
```

Щоб отримати **числа**, потрібно явне перетворення:
```python
int(args[1]) + int(args[2])  # → 10 + 20 = 30
```

### Безпечна обробка аргументів
```python
import sys

if __name__ == "__main__":
    if len(sys.argv) != 3:
        print(f"Використання: python {sys.argv[0]} <число1> <число2>")
        sys.exit(1)

    if not sys.argv[1].isdigit() or not sys.argv[2].isdigit():
        print("Помилка: обидва аргументи мають бути цілими числами")
        sys.exit(1)

    total = int(sys.argv[1]) + int(sys.argv[2])
    print("Сума:", total)
```

## 🔑 Правило
> `sys.argv` — це список рядків. Завжди перетворюйте типи явно.  
> Завжди перевіряйте `len(sys.argv)` перед зверненням до індексів.
"""))

        try:
            submit_task(STUDENT_NAME, _LESSON_ID, "Task_sys_argv")
        except Exception:
            print("ℹ️ submit_task() не підключений")

_rev4_btn.on_click(_on_rev4)
display(widgets.VBox([_rev4_btn, _rev4_out]))

---
## 💻 Частина 5: Консольні програми (CLI) з `while True:`

Типова структура CLI-програми:

```python
while True:                          # нескінченний цикл очікування
    raw = input("Введіть: ").strip() # отримуємо ввід

    if raw.lower() == "exit":        # вихід
        break

    if not raw.isdigit():            # валідація
        print("Помилка")
        continue                     # наступна ітерація

    print(int(raw) * 2)              # обробка
```

### Методи рядка для валідації (без `try/except`)

| Метод | Повертає `True` якщо... | Приклади |
|-------|------------------------|----------|
| `.isdigit()` | рядок містить тільки цифри `0-9` | `"42"` ✅, `"-5"` ❌, `"3.14"` ❌ |
| `.isalpha()` | тільки літери | `"Alice"` ✅, `"Al 1"` ❌ |
| `.isalnum()` | цифри і літери | `"ABC123"` ✅ |
| `.strip()` | (видаляє пробіли по краях) | `"  10  "` → `"10"` |

# ✅ Task 5 — Валідація вводу в CLI

### Що виведе Python для кожного вводу?

```python
inputs = ["  42  ", "ten", "0", "-5"]

for raw in inputs:
    cleaned = raw.strip()
    if not cleaned.isdigit():
        print(f"'{raw}' → ПОМИЛКА")
        continue
    print(f"'{raw}' → {int(cleaned) * 2}")
```

> ✏️ Введіть очікувані результати для всіх 4 рядків, наприклад: `84, ПОМИЛКА, 0, ПОМИЛКА`

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prediction_cli = ""

_pt5_w   = widgets.Text(placeholder="84, ПОМИЛКА, 0, ПОМИЛКА",
                         description="Прогноз:", layout=widgets.Layout(width="420px"))
_pt5_btn = widgets.Button(description="Зберегти прогноз", button_style="primary")
_pt5_out = widgets.Output()

def _on_pt5(b):
    global prediction_cli
    if not require_student():
        with _pt5_out:
            clear_output(); print("⛔ Спочатку представтесь")
        return
    val = _pt5_w.value.strip()
    if not val:
        with _pt5_out:
            clear_output(); print("⚠️ Введіть прогноз")
        return
    prediction_cli = val
    _pt5_btn.disabled = True
    with _pt5_out:
        clear_output(); print("✅ Прогноз збережено:", prediction_cli)

_pt5_btn.on_click(_on_pt5)
display(widgets.VBox([_pt5_w, _pt5_btn, _pt5_out]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

_rev5_btn = widgets.Button(description="🐍 Результат та пояснення", button_style="info")
_rev5_out = widgets.Output()

def _on_rev5(b):
    if not require_student():
        with _rev5_out: print("⛔ Спочатку представтесь")
        return
    if not prediction_cli.strip():
        with _rev5_out: print("⚠️ Спочатку зробіть прогноз")
        return
    b.disabled = True
    with _rev5_out:
        print(f"{STUDENT_NAME}, твій прогноз: {prediction_cli}")

        inputs = ["  42  ", "ten", "0", "-5"]
        print("\n🐍 Покроковий результат:")
        results = []
        for raw in inputs:
            cleaned = raw.strip()
            if not cleaned.isdigit():
                label = "ПОМИЛКА"
                print(f"  '{raw}'  →  '{cleaned}'.isdigit() = False  →  ПОМИЛКА")
            else:
                val = int(cleaned) * 2
                label = str(val)
                print(f"  '{raw}'  →  '{cleaned}'.isdigit() = True   →  {val}")
            results.append(label)

        print("\n📌 Результати:", ", ".join(results))

        # Перевірка прогнозу
        pred = prediction_cli.lower()
        correct = all(
            x in pred
            for x in ["84", "помилка", "0"]
        )
        if correct:
            print("\n✅ Правильний прогноз!")
        else:
            print("\n❌ Неправильно. Відповідь: 84, ПОМИЛКА, 0, ПОМИЛКА")

        display(Markdown("""
---

## ✅ Що сталося?

| Ввід | `strip()` | `isdigit()` | Результат |
|------|----------|------------|----------|
| `"  42  "` | `"42"` | ✅ True | `42 * 2 = 84` |
| `"ten"` | `"ten"` | ❌ False | ПОМИЛКА |
| `"0"` | `"0"` | ✅ True | `0 * 2 = 0` |
| `"-5"` | `"-5"` | ❌ False | ПОМИЛКА |

### Чому `-5` → ПОМИЛКА?
`isdigit()` повертає `True` **лише для цифр `0-9`**.  
Знак `-` і крапка `.` — не цифри, тому `"-5".isdigit() == False`.

### Як обробити від'ємні числа?
```python
# Варіант 1: lstrip для знаку мінус
cleaned.lstrip('-').isdigit()

# Варіант 2: try/except (для складніших випадків)
try:
    number = int(cleaned)
except ValueError:
    print("Помилка")
```

## 🔑 Правило
> `.strip()` + `.isdigit()` — простий спосіб валідувати **невід'ємні цілі числа**.  
> Для від'ємних, дробових чи складних форматів — використовуйте `try/except ValueError`.
"""))

        try:
            submit_task(STUDENT_NAME, _LESSON_ID, "Task_cli_validation")
        except Exception:
            print("ℹ️ submit_task() не підключений")

_rev5_btn.on_click(_on_rev5)
display(widgets.VBox([_rev5_btn, _rev5_out]))

---
# 📝 Exam — Тест

Питання завантажуються з сервера.  
Натисніть кнопку нижче — з'являться питання трьох рівнів складності.  
Кожна відповідь перевіряється автоматично.

In [ ]:
# ── EXAM — питання з Google Sheets ────────────────────────────────────────────
from tools.exam_engine import make_exam_widget

display(make_exam_widget("lesson_05_exam", lambda: STUDENT_NAME, require_student))
